# Отчёт: распознавание изображений (Teachable Machine, Pocket AutoML, API)

В отчёте четыре задания. Код общий: сначала загружаем функции (ячейки ниже), потом используем их для каждой модели.

**Как запускать:** из папки `teachable_machine` (рядом должны лежать `models/`, `flowers_validation/`, `hot_dog_validation/` и т.д.), затем *Restart & Run All* и **Ctrl+S**, чтобы картинки сохранились в файле.


In [1]:
import json
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from contextlib import contextmanager

import h5py
import keras
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from keras.layers import GlobalAveragePooling2D
from keras.src.legacy.saving import legacy_h5_format, saving_utils
import keras.src.layers.layer as keras_layer_module

np.set_printoptions(suppress=True)


@contextmanager
def pick_vector_from_pair():
    orig_call = keras_layer_module.Layer.__call__

    def patched(self, *args, **kwargs):
        inputs = getattr(self, "inputs", None)
        if (
            isinstance(inputs, list)
            and len(inputs) == 1
            and len(args) == 1
            and isinstance(args[0], (list, tuple))
            and len(args[0]) == 2
        ):
            want_rank = len(inputs[0].shape)
            same_rank = [t for t in args[0] if len(t.shape) == want_rank]
            if same_rank:
                t = same_rank[0]
            else:
                t = args[0][0]
                if want_rank == 2 and len(t.shape) == 4:
                    t = GlobalAveragePooling2D()(t)
            args = (t,)
        return orig_call(self, *args, **kwargs)

    keras_layer_module.Layer.__call__ = patched
    try:
        yield
    finally:
        keras_layer_module.Layer.__call__ = orig_call


def clean_config(obj):
    if isinstance(obj, dict):
        obj.pop("groups", None)
        for v in obj.values():
            clean_config(v)
    elif isinstance(obj, list):
        for item in obj:
            clean_config(item)


def load_tm_model(h5_path):
    with h5py.File(h5_path, "r") as f:
        model_config = f.attrs.get("model_config")
        if isinstance(model_config, bytes):
            model_config = model_config.decode("utf-8")
        config_dict = json.loads(model_config)
        clean_config(config_dict)

        with pick_vector_from_pair():
            model = saving_utils.model_from_config(config_dict)
        legacy_h5_format.load_weights_from_hdf5_group(f["model_weights"], model)
    return model


def iter_leaf_layers(layers_list):
    for layer in layers_list:
        if isinstance(layer, keras.Sequential):
            yield from iter_leaf_layers(layer.layers)
        else:
            yield layer


def build_inference_model(loaded):
    inp = keras.Input(shape=(224, 224, 3))
    x = inp
    for layer in iter_leaf_layers(loaded.layers):
        if isinstance(x, (list, tuple)):
            x = x[0]
        if isinstance(layer, keras.layers.Dense):
            want_rank = 2
        else:
            try:
                want_rank = len(layer.inputs[0].shape)
            except Exception:
                want_rank = None
        if want_rank == 2 and len(x.shape) == 4:
            x = GlobalAveragePooling2D()(x)
        x = layer(x)
    if isinstance(x, (list, tuple)):
        x = x[0]
    return keras.Model(inp, x)


def load_model_and_labels(model_path, labels_path):
    model = build_inference_model(load_tm_model(model_path))
    with open(labels_path, "r", encoding="utf-8") as f:
        class_names = [line.strip() for line in f if line.strip()]
    print("   Модель загружена:", model_path)
    print("   Вход:", model.input_shape, "| выход:", model.output_shape)
    print("   Классы:", class_names)
    return model, class_names


I0000 00:00:1795378065.649079  174250 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795378066.930485  174250 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
import os

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps


def predict_image(model, class_names, image_path):
    data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
    image = Image.open(image_path).convert("RGB")
    image = ImageOps.fit(image, (224, 224), Image.Resampling.LANCZOS)
    data[0] = np.asarray(image).astype(np.float32) / 127.5 - 1

    prediction = model.predict(data, verbose=0)
    index = int(np.argmax(prediction))
    name = class_names[index]
    class_name = name.split(" ", 1)[1] if " " in name else name
    return image, index, class_name, float(prediction[0][index])


def evaluate_folder(model, class_names, folder_path, expected_class_index, show_count=10):
    files = sorted(
        f for f in os.listdir(folder_path)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
    )
    total = len(files)
    correct = 0
    answers = [0] * len(class_names)  # сколько раз модель выбрала каждый класс

    print(f"\n=== Папка: {folder_path} (фото: {total}) ===")
    show_count = min(show_count, total, 10)
    if show_count > 0:
        fig, axes = plt.subplots(2, 5, figsize=(16, 8))
        axes = axes.flatten()
        for ax in axes:
            ax.axis("off")

    for i, file in enumerate(files):
        img, pred_index, pred_name, conf = predict_image(model, class_names, os.path.join(folder_path, file))
        answers[pred_index] += 1
        is_correct = pred_index == expected_class_index
        correct += is_correct

        if i < show_count:
            axes[i].imshow(img)
            axes[i].set_title(f"{pred_name}\n({conf * 100:.1f}%)",
                              color="green" if is_correct else "red", fontsize=11)

    if show_count > 0:
        plt.tight_layout()
        plt.savefig(f"result_{os.path.basename(folder_path)}.png", dpi=100)  # на случай, если редактор не хранит картинки
        plt.show()

    print(f"Правильно угадано: {correct} из {total} ({correct / total * 100 if total else 0:.1f}%)")
    names = [n.split(" ", 1)[1] if " " in n else n for n in class_names]
    print("Что отвечала модель:", ", ".join(f"{n}: {c}" for n, c in zip(names, answers)))
    print("-" * 60)
    return correct, total


def check_model(model, class_names, folders):
    all_correct = all_total = 0
    for folder, idx in folders:
        ok, n = evaluate_folder(model, class_names, folder, idx)
        all_correct += ok
        all_total += n
    print(f"ИТОГО: {all_correct} из {all_total} ({all_correct / all_total * 100:.1f}%)")


## Задание 1. Модель «хот-дог / не хот-дог» (Teachable Machine)

**Ход работы.** В Teachable Machine создано два класса (`hotdog` и `nohoddog`), модель обучена на тренировочных изображениях. Работа модели проверена в разделе «Посмотреть»: по 10 фото из `hot_dog_validation` и `not_hot_dog_validation`. Модель сохранена в формате Keras: `models/keras_model.h5` и `models/labels.txt`.

![](images/acuracy_epoch_teachblemachine.png)
![](images/acuracy_teachblemachine.png)

**Вывод по 10 фото хот-догов (Teachable Machine):** 
![](result_hot_dog_validation.png)
**Вывод по 10 фото «не хот-догов» (Teachable Machine):**
![](result_not_hot_dog_validation.png)

### Загрузка модели в Jupyter

Фрагмент кода, который предлагает Teachable Machine:

```python
from keras.models import load_model  # TensorFlow is required for Keras to work
from PIL import Image, ImageOps  # Install pillow instead of PIL
import numpy as np

np.set_printoptions(suppress=True)

# Load the model
model = load_model("keras_Model.h5", compile=False)

# Load the labels
class_names = open("labels.txt", "r").readlines()

data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
image = Image.open("<IMAGE_PATH>").convert("RGB")
image = ImageOps.fit(image, (224, 224), Image.Resampling.LANCZOS)
image_array = np.asarray(image)
normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1
data[0] = normalized_image_array

prediction = model.predict(data)
index = np.argmax(prediction)
class_name = class_names[index]
confidence_score = prediction[0][index]

print("Class:", class_name[2:], end="")
print("Confidence Score:", confidence_score)
```

На моём окружении (Python 3.14, Keras 3) этот код не запускается: `load_model` падает с ошибкой `Unrecognized keyword arguments ... DepthwiseConv2D: {'groups': 1}`. Поэтому загрузку я переписал (функции в первой ячейке):
1. читаю конфиг модели из `.h5` и удаляю из него ключ `groups`;
2. собираю модель и подгружаю веса вручную; на время сборки подменяю `Layer.__call__`, потому что Keras 3 неправильно соединяет вложенные части модели;
3. разворачиваю вложенные блоки в один плоский список слоёв и собираю чистую модель `Input(224, 224, 3) -> слои -> выход`.

Предобработка фото такая же, как в Teachable Machine: обрезка до 224×224 и нормализация `пиксель / 127.5 - 1`.


In [3]:
hotdog_model, hotdog_classes = load_model_and_labels("models/keras_model.h5", "models/labels.txt")

check_model(hotdog_model, hotdog_classes, [
    ("hot_dog_validation", 0),
    ("not_hot_dog_validation", 1),
])


/home/yaekitsune/Projects/MOIBD_college_learning_machine_learn/teachable_machine/.venv/lib/python3.14/site-packages/keras/src/layers/core/dense.py:97: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


✅ Модель загружена: models/keras_model.h5
   Вход: (None, 224, 224, 3) | выход: (None, 2)
   Классы: ['0 hotdog 1', '1 nohoddog 2']
=== Папка: hot_dog_validation (фото: 250) ===


Правильно угадано: 210 из 250 (84.0%)
Что отвечала модель: hotdog 1: 210, nohoddog 2: 40
------------------------------------------------------------
=== Папка: not_hot_dog_validation (фото: 250) ===


Правильно угадано: 224 из 250 (89.6%)
Что отвечала модель: hotdog 1: 26, nohoddog 2: 224
------------------------------------------------------------
ИТОГО: 434 из 500 (86.8%)


### Результат проверки (задание 1)

| Папка | Верно | Точность |
|---|---|---|
| hot_dog_validation | 210 из 250 | 84,0% |
| not_hot_dog_validation | 224 из 250 | 89,6% |
| **Всего** | **434 из 500** | **86,8%** |

**Вывод.** Модель в целом справляется: на 500 фото, которых она не видела, верно определено около 87%. Хот-доги она находит хуже (84%), чем «не хот-доги» (89,6%): 40 хот-догов пропущено, а «не хот-догом» ошибочно названо 26 других фото.

## Задание 2. Модель из трёх классов: цветы (Teachable Machine)

**Данные.** Датасет с Kaggle *Flowers Recognition* (автор alxmamaev). Взято 3 класса: **ромашка, роза, подсолнух**. Часть фото загружена в обучение, ещё по 50 фото на класс отложено для проверки (`flowers_validation`) и модели не показывалось.

**Точность на каждый класс по данным Teachable Machine** (встроенный тест, по 23 фото на класс):

| Класс | Точность | Верно |
|---|---|---|
| ромашка | 0,87 | 20 из 23 |
| Роза | 0,83 | 19 из 23 |
| Подсолнух | 1,00 | 23 из 23 |

Матрица ошибок: ромашка принята за розу 3 раза; роза за ромашку 2 раза и за подсолнух 2 раза; подсолнух определён без ошибок. Всего верно 62 из 69 (около 90%).

![](images/accuracy_epoch_teachble_flowers.png)
![](images/accuracy_teachble_flowers.png)

Модель сохранена: `models/flowers_model.h5` и `models/flowers_labels.txt`.


In [4]:
flowers_model, flowers_classes = load_model_and_labels("models/flowers_model.h5", "models/flowers_labels.txt")

check_model(flowers_model, flowers_classes, [
    ("flowers_validation/daisy", 0),
    ("flowers_validation/rose", 1),
    ("flowers_validation/sunflower", 2),
])


✅ Модель загружена: models/flowers_model.h5
   Вход: (None, 224, 224, 3) | выход: (None, 3)
   Классы: ['0 ромашка', '1 Роза', '2 Подсолнух']
=== Папка: flowers_validation/daisy (фото: 50) ===


Правильно угадано: 41 из 50 (82.0%)
Что отвечала модель: ромашка: 41, Роза: 5, Подсолнух: 4
------------------------------------------------------------
=== Папка: flowers_validation/rose (фото: 50) ===


Правильно угадано: 49 из 50 (98.0%)
Что отвечала модель: ромашка: 0, Роза: 49, Подсолнух: 1
------------------------------------------------------------
=== Папка: flowers_validation/sunflower (фото: 50) ===


Правильно угадано: 49 из 50 (98.0%)
Что отвечала модель: ромашка: 1, Роза: 0, Подсолнух: 49
------------------------------------------------------------
ИТОГО: 139 из 150 (92.7%)


### Результат проверки (задание 2)

Проверка на 150 новых фото (по 50 на класс):

| Класс | Верно | Точность |
|---|---|---|
| ромашка | 41 из 50 | 82,0% |
| Роза | 49 из 50 | 98,0% |
| Подсолнух | 49 из 50 | 98,0% |
| **Всего** | **139 из 150** | **92,7%** |

**Вывод.** Модель определяет неизвестные ей цветы примерно в 93% случаев. Самый слабый класс: ромашка (82%), он же был самым слабым в тесте Teachable Machine. Розы и подсолнухи определяются почти безошибочно. Оценка на 150 фото надёжнее встроенного теста Teachable Machine (там всего 23 фото на класс, поэтому у розы получилось 0,83, а на 50 фото 0,98).


## Задание 3. Pocket AutoML (Android)

**Ход работы.** В приложении создана задача с тремя классами: **подсолнухи, розы, ромашки**. В каждый класс добавлено по 20 фото (всего 60), модель обучена на вкладке «Модель». Затем на вкладке «Распознать» модели показаны новые фото, ответы подтверждались или исправлялись, после чего модель обучена повторно.

**Данные на экране «Модель» (на момент скриншота):** тренировочных изображений 68 (60 исходных + 8 добавленных при подтверждении), всего распознаваний 12, обратная связь (подтверждено / всего) 10/11, точность от 61,52% до 97,91% (95% достоверность). Диапазон широкий, потому что проверок всего 11.

### Скриншоты распознавания

**Подсолнухи**

![](<images_questinon/Подсолнухи_вывод.png>)
![](<images_questinon/Подсолнухи_вывод2.png>)
![](<images_questinon/Подсолнухи_вывод3.png>)

**Розы**

![](<images_questinon/Роза_вывод.png>)
![](<images_questinon/Роза_вывод2.png>)
![](<images_questinon/Роза_вывод3.png>)
![](<images_questinon/Роза_вывод_4.png>)

**Ромашки**

![](<images_questinon/Ромашки_вывод.png>)
![](<images_questinon/Ромашки_вывод2.png>)

Ошибка модели и исправление:

![](<images_questinon/Ромашки_неправильный_вывод.png>)
![](<images_questinon/Ромашки_неправильный_вывод_исправление.png>)

### Вывод (задание 3)

Из 11 ответов с обратной связью 10 модель дала верно, один пришлось исправить (ромашка).

**Возможные причины ошибок** (проверь на своих скриншотах и оставь только то, что подтверждается):
- в обучающей выборке попались неподходящие фото: например, в «розах» есть повозка с лошадьми и садовый пейзаж, где розы почти не видны; в «подсолнухах» автомобиль на переднем плане;
- цветок на фото мелкий или занимает малую часть кадра;
- ромашки и розы бывают похожи по цвету и фону;
- фото в каждом классе мало (20), и 11 проверок недостаточно для точной оценки.


## Задание 4. API и приложение с интерфейсом

**API** написан на двух фреймворках (Flask и FastAPI), оба работают одинаково:
- запрос: `POST /predict`, в теле запроса байты картинки;
- ответ: JSON с номером класса, названием и точностью, например `{"index": 0, "class": "ромашка", "confidence": 0.93}`;
- при ошибке (не картинка, пустой запрос) возвращается код 400 и текст ошибки.

Модель (из задания 2) загружается один раз при запуске сервера, код загрузки тот же, что выше.

**Запуск** (из папки `api/`):
```
python api_flask.py       # или: python api_fastapi.py
```
**Приложение с интерфейсом** на Streamlit (запускается в браузере) и запасной вариант на tkinter. Пользователь выбирает фото, оно отправляется на API, на экране показывается класс и точность.
```
streamlit run client_streamlit.py
```


Скриншоты:

![dasd](images_questinon/Роза2.png)
![dasd](images_questinon/ромашка2.png)
![dasd](images_questinon/Розы.png)
![dasd](images_questinon/Подсолнухи.png)
![dasd](images_questinon/Подсолнухи1.png)

**Вывод.** Модель обёрнута в API, которое принимает картинку и возвращает класс и точность, а приложение с интерфейсом показывает результат пользователю.
